In [ ]:
import requests
from datetime import datetime, timedelta
import time


symbols = ["EUR", "NOK", "SEK", "PLN", "RON", "DKK", "CZK"]


start_date = "2026-01-3"
end_date = "2026-01-05"


api_start = (datetime.strptime(start_date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
api_end = end_date

all_data = []


for base in symbols:
    targets = [s for s in symbols if s != base]
    to_param = ",".join(targets)
    url = f"https://api.frankfurter.app/{api_start}..{api_end}?from={base}&to={to_param}"

    resp = requests.get(url)
    if resp.status_code == 200:
        all_data.append({
            "base": base,
            "targets": targets,
            "data": resp.json()
        })
    else:
        all_data.append({"base": base, "error": resp.status_code})

    
    time.sleep(0.2)

print(f"{len(all_data)}")

all_data[0] if all_data else None

تعداد پاسخ‌های دریافت شده برای بیس‌ها: 7


{'base': 'EUR',
 'targets': ['NOK', 'SEK', 'PLN', 'RON', 'DKK', 'CZK'],
 'data': {'amount': 1.0,
  'base': 'EUR',
  'start_date': '2026-01-02',
  'end_date': '2026-01-05',
  'rates': {'2026-01-02': {'CZK': 24.177,
    'DKK': 7.4694,
    'NOK': 11.7985,
    'PLN': 4.2123,
    'RON': 5.0895,
    'SEK': 10.8085},
   '2026-01-05': {'CZK': 24.195,
    'DKK': 7.4695,
    'NOK': 11.7795,
    'PLN': 4.2178,
    'RON': 5.0875,
    'SEK': 10.787}}}}

In [14]:
all_data[0]

{'pair': 'EUR/NOK',
 'data': {'amount': 1.0,
  'base': 'EUR',
  'start_date': '2026-01-02',
  'end_date': '2026-01-05',
  'rates': {'2026-01-02': {'NOK': 11.7985}, '2026-01-05': {'NOK': 11.7795}}}}

In [10]:
import sqlite3
import os

# go one level up to fx-pipeline
os.chdir("..")

# create database and run schema
conn = sqlite3.connect("C:\\Users\\hosse\\Desktop\\fx-pipeline\\db\\fx_dwh.sqlite")

with open("C:\\Users\\hosse\\Desktop\\fx-pipeline\\db\\schema.sql", "r") as f:
    conn.executescript(f.read())

conn.commit()
conn.close()

print("Database created successfully!")

Database created successfully!


In [4]:
import os
print(os.getcwd())

c:\Users\hosse\Desktop


In [12]:
import sqlite3
import os
from itertools import permutations

base_dir = r"C:\Users\hosse\Desktop\fx-pipeline"

conn = sqlite3.connect(f"{base_dir}/db/fx_dwh.sqlite")

with open(f"{base_dir}/db/schema.sql", "r") as f:
    conn.executescript(f.read())

# seed dim_currency_pair
currencies = ['EUR', 'NOK', 'SEK', 'PLN', 'RON', 'DKK', 'CZK']
pairs = [(a, b, f"{a}/{b}") for a, b in permutations(currencies, 2)]

conn.executemany("""
    INSERT OR IGNORE INTO dim_currency_pair (base, quote, pair_label)
    VALUES (?, ?, ?)
""", pairs)

conn.commit()
conn.close()

print(f"Database created with {len(pairs)} currency pairs!")

Database created with 42 currency pairs!


In [14]:
import sqlite3

base_dir = r"C:\Users\hosse\Desktop\fx-pipeline"
conn = sqlite3.connect(f"{base_dir}/db/fx_dwh.sqlite")

# check tables
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("Tables:", tables)

# check pairs count
count = conn.execute("SELECT COUNT(*) FROM dim_currency_pair").fetchone()
print("Currency pairs:", count[0])

# show first 5 pairs
pairs = conn.execute("SELECT * FROM dim_currency_pair LIMIT 5").fetchall()
for p in pairs:
    print(p)

conn.close()

Tables: [('dim_currency_pair',), ('sqlite_sequence',), ('fact_fx_rates',)]
Currency pairs: 42
(1, 'EUR', 'NOK', 'EUR/NOK')
(2, 'EUR', 'SEK', 'EUR/SEK')
(3, 'EUR', 'PLN', 'EUR/PLN')
(4, 'EUR', 'RON', 'EUR/RON')
(5, 'EUR', 'DKK', 'EUR/DKK')


In [17]:
import sqlite3
import os

DB_PATH = r"C:\Users\hosse\Desktop\fx-pipeline\db\fx_dwh.sqlite"
conn = sqlite3.connect(DB_PATH)

# total rows
total = conn.execute("SELECT COUNT(*) FROM fact_fx_rates").fetchone()[0]
print(f"Total rows: {total}")

# check for duplicates
duplicates = conn.execute("""
    SELECT pair_id, date, COUNT(*) as count
    FROM fact_fx_rates
    GROUP BY pair_id, date
    HAVING COUNT(*) > 1
""").fetchall()

if duplicates:
    print(f"Duplicates found: {duplicates}")
else:
    print("No duplicates found!")

conn.close()

Total rows: 14658
No duplicates found!


In [20]:
DB_PATH = r"C:\Users\hosse\Desktop\fx-pipeline\db\fx_dwh.sqlite"
conn = sqlite3.connect(DB_PATH)

# 1. یه pair در یه تاریخ مشخص
print("=== 1. EUR/NOK on 2025-06-15 ===")
rows = conn.execute("""
    SELECT f.date, p.pair_label, f.rate
    FROM fact_fx_rates f
    JOIN dim_currency_pair p ON f.pair_id = p.id
    WHERE p.pair_label = 'EUR/NOK'
      AND f.date = '2025-06-15'
""").fetchall()
for r in rows:
    print(r)

=== 1. EUR/NOK on 2025-06-15 ===


In [21]:
# 2. daily change برای یه ماه
print("\n=== 2. EUR/NOK daily change - Jan 2024 ===")
rows = conn.execute("""
    SELECT f.date, f.rate, f.prev_day_rate, ROUND(f.daily_change_pct, 4) AS daily_change
    FROM fact_fx_rates f
    JOIN dim_currency_pair p ON f.pair_id = p.id
    WHERE p.pair_label = 'EUR/NOK'
      AND f.date BETWEEN '2025-01-01' AND '2025-01-31'
    ORDER BY f.date
""").fetchall()
for r in rows:
    print(r)


=== 2. EUR/NOK daily change - Jan 2024 ===
('2025-01-02', 11.7173, 11.795, -0.6588)
('2025-01-03', 11.713, 11.7173, -0.0367)
('2025-01-06', 11.7095, 11.713, -0.0299)
('2025-01-07', 11.7385, 11.7095, 0.2477)
('2025-01-08', 11.738, 11.7385, -0.0043)
('2025-01-09', 11.7605, 11.738, 0.1917)
('2025-01-10', 11.757, 11.7605, -0.0298)
('2025-01-13', 11.709, 11.757, -0.4083)
('2025-01-14', 11.7195, 11.709, 0.0897)
('2025-01-15', 11.7035, 11.7195, -0.1365)
('2025-01-16', 11.6965, 11.7035, -0.0598)
('2025-01-17', 11.759, 11.6965, 0.5343)
('2025-01-20', 11.7665, 11.759, 0.0638)
('2025-01-21', 11.817, 11.7665, 0.4292)
('2025-01-22', 11.7525, 11.817, -0.5458)
('2025-01-23', 11.7305, 11.7525, -0.1872)
('2025-01-24', 11.7495, 11.7305, 0.162)
('2025-01-27', 11.8095, 11.7495, 0.5107)
('2025-01-28', 11.7785, 11.8095, -0.2625)
('2025-01-29', 11.7785, 11.7785, 0.0)
('2025-01-30', 11.7615, 11.7785, -0.1443)
('2025-01-31', 11.7373, 11.7615, -0.2058)


In [22]:
# 3. YTD آخر سال
print("\n=== 3. YTD at end of 2025 - top 5 ===")
rows = conn.execute("""
    SELECT p.pair_label, f.rate, f.year_start_rate, ROUND(f.ytd_pct, 4) AS ytd
    FROM fact_fx_rates f
    JOIN dim_currency_pair p ON f.pair_id = p.id
    WHERE f.date = '2025-12-31'
    ORDER BY f.ytd_pct DESC
    LIMIT 5
""").fetchall()
for r in rows:
    print(r)

conn.close()


=== 3. YTD at end of 2025 - top 5 ===
('SEK/RON', 0.47099, 0.43549, 8.1517)
('SEK/NOK', 1.0944, 1.0258, 6.6875)
('CZK/RON', 0.21029, 0.19757, 6.4382)
('SEK/DKK', 0.69019, 0.65296, 5.7017)
('SEK/EUR', 0.09241, 0.08755, 5.5511)
